# Tiny Chat 42M - Kaggle GPU Training

Kaggle GPU notebook for the tiny_chat project.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU in Kaggle: Settings > Accelerator > GPU')
print('GPU:', torch.cuda.get_device_name(0))

## Clone and install

The Git LFS dataset is skipped. The next cell downloads data directly from Hugging Face.

In [ ]:
PROJECT_DIR = Path('/kaggle/working/ai')
REPO_URL = 'https://github.com/abdelqalzeinn-cmyk/ai.git'
if not (PROJECT_DIR / 'pyproject.toml').exists():
    os.environ['GIT_LFS_SKIP_SMUDGE'] = '1'
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('Installed:', PROJECT_DIR)

## Download and prepare Hugging Face data

In [ ]:
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'tiny_chat.data'], check=True)
print('Dataset ready')

In [ ]:
STEPS = 5000
BATCH_SIZE = 16
MAX_CHARS = 20000000
DEVICE = 'cuda'
CHECKPOINT = '/kaggle/working/chat_model.pt'
print({'steps': STEPS, 'batch_size': BATCH_SIZE, 'max_chars': MAX_CHARS})

## GPU smoke test

In [ ]:
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'tiny_chat.train', '--steps', '2', '--batch-size', '2', '--max-chars', '100000', '--device', DEVICE, '--checkpoint', CHECKPOINT], check=True)

## Train

Save the Kaggle notebook version after training to preserve the checkpoint.

In [ ]:
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'tiny_chat.train', '--steps', str(STEPS), '--batch-size', str(BATCH_SIZE), '--max-chars', str(MAX_CHARS), '--device', DEVICE, '--checkpoint', CHECKPOINT], check=True)

## Test a response

This cell generates one response without interactive terminal input.

In [ ]:
from tiny_chat.config import ModelConfig
from tiny_chat.model import ChatModel
from tiny_chat.tokenizer import ByteBPETokenizer
saved = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
tokenizer = ByteBPETokenizer.from_state_dict(saved['tokenizer'])
model = ChatModel(ModelConfig(**saved['config'])).to(DEVICE)
model.load_state_dict(saved['model'])
model.eval()
prompt = 'User: hi\nAssistant:'
inputs = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=DEVICE)
with torch.no_grad():
    output = model.generate(inputs, max_new_tokens=80, temperature=0.5, top_k=40, top_p=0.9, repetition_penalty=1.1)
print(tokenizer.decode(output[0].tolist()))